In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.shape)

df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Check for missing values
print(df.dtypes)
print()
print("Missing Values:", df.isnull().sum())
print("Duplicates:", df['customerID'].duplicated().sum())




customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Missing Values: customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
Payment

In [3]:
# TotalCharges fix
bad = pd.to_numeric(df['TotalCharges'], errors='coerce').isnull()

print("Bad Values:", bad.sum())

df.loc[bad, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]


Bad Values: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [4]:
# Clean Data

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop CustomerID column
df = df.drop('customerID', axis=1)

# Convert target to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("TotalCharges type:", df['TotalCharges'].dtype)
print("Shape:", df.shape)
print()
print(df['Churn'].value_counts())


TotalCharges type: float64
Shape: (7043, 20)

Churn
0    5174
1    1869
Name: count, dtype: int64


In [5]:
# How churn varies by contract type
print(df.groupby('Contract')['Churn'].mean().round(3))
print()

# By InternetService
print(df.groupby('InternetService')['Churn'].mean().round(3))
print()

# Average tenure churners vs non-churners
print(df.groupby('Churn')['tenure'].mean().round(1))
print()

# Average MonthlyCharges churners vs non-churners
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

Contract
Month-to-month    0.427
One year          0.113
Two year          0.028
Name: Churn, dtype: float64

InternetService
DSL            0.190
Fiber optic    0.419
No             0.074
Name: Churn, dtype: float64

Churn
0    37.6
1    18.0
Name: tenure, dtype: float64

Churn
0    61.27
1    74.44
Name: MonthlyCharges, dtype: float64


In [6]:
# Convert text columns to numeric
df_encoded = pd.get_dummies(df, drop_first=True)

print(df_encoded.shape)


(7043, 31)


In [7]:
#Split the data into training and testing sets

from sklearn.model_selection import train_test_split

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# 80/20 split for training and testing
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Second split
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
                                                  

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



Train: (4225, 30)
Validation: (1409, 30)
Test: (1409, 30)


In [8]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled")


Scaled


In [9]:
# Logistic Regression Model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, lr_pred))
print()
print("Confusion Matrix:\n", confusion_matrix(y_val, lr_pred))
print()
print("Classification Report:\n", classification_report(y_val, lr_pred))



Accuracy: 0.7459190915542938

Confusion Matrix:
 [[756 279]
 [ 79 295]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



In [10]:
# Random Forest Model

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train_scaled, y_train)

rf_pred = rf.predict(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, rf_pred))
print()
print("Confusion Matrix:\n", confusion_matrix(y_val, rf_pred))
print()
print("Classification Report:\n", classification_report(y_val, rf_pred))



Accuracy: 0.7877927608232789

Confusion Matrix:
 [[941  94]
 [205 169]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.91      0.86      1035
           1       0.64      0.45      0.53       374

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.77      0.79      0.77      1409



In [11]:
# Gradient Boosting Model

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_scaled, y_train)

gb_pred = gb.predict(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, gb_pred))
print()
print("Confusion Matrix:\n", confusion_matrix(y_val, gb_pred))
print()
print("Classification Report:\n", classification_report(y_val, gb_pred))


Accuracy: 0.8034066713981547

Confusion Matrix:
 [[947  88]
 [189 185]]

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.91      0.87      1035
           1       0.68      0.49      0.57       374

    accuracy                           0.80      1409
   macro avg       0.76      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [12]:
import pickle
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

pipeline.fit(X_train, y_train)

# Evaluate the best model on the test set

test_pred = pipeline.predict(X_test)

print("TEST RESULTS")
print("Accuracy:", accuracy_score(y_test, test_pred))
print()
print("Confusion Matrix:\n", confusion_matrix(y_test, test_pred))
print()
print("Classification Report:\n", classification_report(y_test, test_pred))

# Save the model to a file
with open('churn_model.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

print("Model Saved to pkl")




TEST RESULTS
Accuracy: 0.7402413058907026

Confusion Matrix:
 [[750 285]
 [ 81 293]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Model Saved to pkl


In [13]:
print(list(X_train.columns))

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [14]:
%pip install --force-reinstall --no-cache-dir "sagemaker>=2,<3"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 101.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 191.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 148.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 107.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 398.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 313.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 129.8 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=1935465b4df8eff153e4339490d3d4c83ad5e126092666e6b23ce92fc5f3c832
  Stored in directory: /tmp/pip-ephem-wheel-cache-q30klwew/wheels/1f/be/48/13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfully built an

In [15]:
import numpy as np
import pandas as pd
import sagemaker
from sagemaker import get_execution_role

session = sagemaker.Session()
role = get_execution_role()
bucket = session.default_bucket()
prefix = 'sagemaker/telco-churn'


df_sm = pd.concat([df_encoded['Churn'], df_encoded.drop(columns=['Churn'])], axis=1)
df_sm = df_sm.astype(float)

shuffled = df_sm.sample(frac=1, random_state=42).reset_index(drop=True)
split_at = int(0.8 * len(shuffled))
train_sm = shuffled.iloc[:split_at]
val_sm = shuffled.iloc[split_at:]

train_sm.to_csv('train.csv', header=False, index=False)
val_sm.to_csv('validation.csv', header=False, index=False)

train_s3 = session.upload_data('train.csv', bucket=bucket, key_prefix=f'{prefix}/train')
val_s3 = session.upload_data('validation.csv', bucket=bucket, key_prefix=f'{prefix}/val')

print("Uploaded to:", bucket)
print("Shape:", train_sm.shape)

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploaded to: amazon-sagemaker-365698175161-us-east-2-cu2ccb8aeza2sp
Shape: (5634, 31)


In [18]:
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator

container = image_uris.retrieve('linear-learner', session.boto_region_name)

session_clean = sagemaker.Session(sagemaker_config={"SchemaVersion": "1.0"})

estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=session_clean
)

estimator.set_hyperparameters(
    feature_dim=30,
    predictor_type='binary_classifier',
    mini_batch_size=100,
    binary_classifier_model_selection_criteria='f_beta',
    target_recall=0.8
)

s3_train = TrainingInput(s3_data=train_s3, content_type='text/csv')
s3_val = TrainingInput(s3_data=val_s3, content_type='text/csv')

estimator.fit({'train': s3_train, 'validation': s3_val})

predictor = estimator.deploy(initial_instance_count=1, instance_type='ml.m5.large')

print("Deployed Successfully")

2026-08-18 07:18:44 Starting - Starting the training job...
2026-08-18 07:19:16 Downloading - Downloading input data......
2026-08-18 07:20:01 Downloading - Downloading the training image.........
2026-08-18 07:21:32 Training - Training image download completed. Training in progress.Docker entrypoint called with argument(s): train
Running default environment configuration script
[08/18/2026 07:21:36 INFO 140442078832448] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 'auto', 'margin': '1.0', 'quantile': '0.5', 'loss_insensitivity': '

In [20]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

sample = val_sm.iloc[0, 1:].values.tolist()
actual = val_sm.iloc[0, 0]

result = predictor.predict(sample)
print("Actual:", actual)
print("Prediction:", result)


Actual: 1.0
Prediction: {'predictions': [{'score': 0.36609625816345215, 'predicted_label': 1}]}


In [22]:
predictor.delete_endpoint()
print("Endpoint deleted")

Endpoint deleted
